In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/scodepy/customer-support-intent-dataset/Bitext_Sample_Customer_Service_Training_Dataset.csv
/kaggle/input/datasets/scodepy/customer-support-intent-dataset/Bitext_Sample_Customer_Service_Validation_Dataset.csv
/kaggle/input/datasets/scodepy/customer-support-intent-dataset/Bitext_Sample_Customer_Service_Testing_Dataset.csv


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("scodepy/customer-support-intent-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/scodepy/customer-support-intent-dataset


In [3]:
import pandas as pd
from datasets import Dataset

# Load data
train = pd.read_csv("/kaggle/input/datasets/scodepy/customer-support-intent-dataset/Bitext_Sample_Customer_Service_Testing_Dataset.csv")
val = pd.read_csv("/kaggle/input/datasets/scodepy/customer-support-intent-dataset/Bitext_Sample_Customer_Service_Training_Dataset.csv")
test = pd.read_csv("/kaggle/input/datasets/scodepy/customer-support-intent-dataset/Bitext_Sample_Customer_Service_Validation_Dataset.csv")

print("Columns:", train.columns.tolist())
print(f"\nTrain: {len(train)}, Val: {len(val)}, Test: {len(test)}")
print(f"\nIntent categories: {train['intent'].nunique()}")
print(train['intent'].value_counts())

# See sample
print("\n--- Sample ---")
print(train[['utterance', 'intent']].head(3))

Columns: ['utterance', 'intent', 'category', 'tags']

Train: 818, Val: 6539, Test: 818

Intent categories: 27
intent
contact_human_agent         42
change_shipping_address     41
change_order                39
delivery_period             38
check_refund_policy         37
switch_account              36
edit_account                35
delivery_options            32
review                      32
set_up_shipping_address     32
check_invoice               31
track_order                 31
get_invoice                 31
place_order                 30
delete_account              29
get_refund                  28
track_refund                28
registration_problems       27
payment_issue               27
check_cancellation_fee      26
check_payment_methods       26
create_account              25
cancel_order                25
contact_customer_service    24
complaint                   23
newsletter_subscription     23
recover_password            20
Name: count, dtype: int64

--- Sample ---
    

In [4]:
# Get unique intents
intent_names = sorted(train['intent'].unique().tolist())
categories = ', '.join(intent_names)

def format_instruction(utterance, intent):
    return f"""### Instruction:
Classify the customer support query into one of these categories: {categories}

### Customer Query:
{utterance}

### Category:
{intent}"""

# Apply to all splits
for df in [train, val, test]:
    df['text'] = df.apply(lambda x: format_instruction(x['utterance'], x['intent']), axis=1)

# Convert to HF Dataset
hf_train = Dataset.from_pandas(train[['text']])
hf_val = Dataset.from_pandas(val[['text']])
hf_test = Dataset.from_pandas(test[['text']])

print(f"Train: {len(hf_train)}, Val: {len(hf_val)}, Test: {len(hf_test)}")
print(f"\nCategories: {len(intent_names)}")
print("\n--- Sample prompt ---")
print(hf_train[0]['text'][:600])

Train: 818, Val: 6539, Test: 818

Categories: 27

--- Sample prompt ---
### Instruction:
Classify the customer support query into one of these categories: cancel_order, change_order, change_shipping_address, check_cancellation_fee, check_invoice, check_payment_methods, check_refund_policy, complaint, contact_customer_service, contact_human_agent, create_account, delete_account, delivery_options, delivery_period, edit_account, get_invoice, get_refund, newsletter_subscription, payment_issue, place_order, recover_password, registration_problems, review, set_up_shipping_address, switch_account, track_order, track_refund

### Customer Query:
I have a question about can


In [5]:
!pip install -U bitsandbytes>=0.46.1

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Qwen2.5-7B-Instruct...")
print("This will download ~15GB. Takes 3-5 minutes...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Qwen tokenizer setup
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Prepare for training
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading Qwen2.5-7B-Instruct...
This will download ~15GB. Takes 3-5 minutes...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [7]:
from transformers import Trainer, TrainingArguments

# Tokenize
def tokenize_function(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_train = hf_train.map(tokenize_function, batched=True)
tokenized_val = hf_val.map(tokenize_function, batched=True)
tokenized_test = hf_test.map(tokenize_function, batched=True)

# Remove text columns
for col in ["text", "__index_level_0__"]:
    if col in tokenized_train.column_names:
        tokenized_train = tokenized_train.remove_columns(col)
    if col in tokenized_val.column_names:
        tokenized_val = tokenized_val.remove_columns(col)
    if col in tokenized_test.column_names:
        tokenized_test = tokenized_test.remove_columns(col)

# Training args
training_args = TrainingArguments(
    output_dir="./voice-intent-qwen25",
    num_train_epochs=3,
    per_device_train_batch_size=2,      # Reduced for 7B model
    gradient_accumulation_steps=8,      # Effective batch = 16
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    args=training_args,
)

print("Starting training...")
trainer.train()

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/6539 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Epoch,Training Loss,Validation Loss
1,0.085787,0.080621
2,0.060114,0.062585
3,0.053635,0.057848


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=156, training_loss=0.5748845718992062, metrics={'train_runtime': 10124.3883, 'train_samples_per_second': 0.242, 'train_steps_per_second': 0.015, 'total_flos': 2.680376489725133e+16, 'train_loss': 0.5748845718992062, 'epoch': 3.0})

In [8]:
# Save model + tokenizer
trainer.model.save_pretrained("./voice-intent-qwen25/final")
tokenizer.save_pretrained("./voice-intent-qwen25/final")
print("Model saved!")

Model saved!


In [9]:
# Save without using trainer
